# 02. Exploratory Data Analysis & Quality Audit
**Course Project:** Cybercrime Complaint Classification Using Machine Learning  
**Dataset Analyzed:** `data/processed/curated_complaints.csv`  

This notebook performs comprehensive exploratory data analysis, checking:
1. Data integrity, null values, and exact duplicates
2. Class distribution and severity of class imbalance
3. Text length distributions across categories
4. Source provenance composition and bias
5. Repetitive template artifacts and n-gram patterns
6. Privacy redaction residual audit

In [ ]:
import os
import re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

DATA_PATH = '../data/processed/curated_complaints.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns.")
df.head(3)

## 1. Data Integrity & Missing Values

In [ ]:
print("--- Missing Values per Column ---")
print(df.isnull().sum())

print("\n--- Duplicate Checks ---")
print("Exact duplicate rows:", df.duplicated().sum())
print("Exact duplicate complaint texts:", df['complaint_text'].duplicated().sum())

## 2. Category Distribution & Class Imbalance

In [ ]:
cat_counts = df['category'].value_counts()
print(cat_counts)

plt.figure(figsize=(10, 5))
colors = ['#d95f02' if c == 'Cyberbullying / Online Harassment' else '#2b5c8f' for c in cat_counts.index]
bars = plt.barh(cat_counts.index, cat_counts.values, color=colors, edgecolor='black', linewidth=0.7)
plt.xlabel('Number of Complaints', fontweight='bold')
plt.title('Category Distribution (Showing Severe Class Imbalance)', fontweight='bold')
plt.gca().invert_yaxis()
for bar in bars:
    w = bar.get_width()
    pct = (w / len(df)) * 100
    plt.text(w + 50, bar.get_y() + bar.get_height() / 2, f"{int(w):,} ({pct:.1f}%)", va='center')
plt.xlim(0, max(cat_counts.values) * 1.18)
plt.tight_layout()
plt.show()

## 3. Complaint Length Metrics by Category

In [ ]:
df['char_length'] = df['complaint_text'].str.len()
df['word_count'] = df['complaint_text'].apply(lambda s: len(str(s).split()))

print(df.groupby('category')[['word_count', 'char_length']].describe())

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    y='category',
    x='word_count',
    order=cat_counts.index,
    palette='Blues_r',
    showmeans=True,
    meanprops={'marker': 'o', 'markerfacecolor': 'red', 'markeredgecolor': 'red'}
)
plt.xlabel('Word Count', fontweight='bold')
plt.ylabel('')
plt.title('Complaint Word Count Distributions by Category', fontweight='bold')
plt.xlim(0, 160)
plt.tight_layout()
plt.show()

## 4. Source Provenance & Source Bias Analysis

In [ ]:
source_crosstab = pd.crosstab(df['category'], df['source_name'])
print(source_crosstab)

source_crosstab.plot(
    kind='barh',
    stacked=True,
    figsize=(10, 5),
    color=['#2b5c8f', '#7570b3'],
    edgecolor='black',
    linewidth=0.6
)
plt.xlabel('Number of Complaints', fontweight='bold')
plt.ylabel('')
plt.title('Source Provenance Composition by Category', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Template & Repetitive Prefix Patterns

In [ ]:
for cat, group in df.groupby('category'):
    prefixes = group['complaint_text'].apply(lambda s: ' '.join(str(s).split()[:4]))
    top_p = Counter(prefixes).most_common(3)
    print(f"\nCategory [{cat}] (Total: {len(group)}):")
    for p, count in top_p:
        print(f"   {count:4d} ({count / len(group) * 100:4.1f}%): \"{p}\"")

## 6. Privacy Redaction Residual Scan

In [ ]:
for tag in ['[USER_REDACTED]', '[URL_REDACTED]', '[EMAIL_REDACTED]', '[PHONE_REDACTED]', '[ACCOUNT_REDACTED]']:
    c = df['complaint_text'].str.contains(re.escape(tag)).sum()
    print(f"{tag:20s}: {c:5,d} occurrences")